# Mission 3 성능 비교

위 메뉴에서 **런타임 → 모두 실행**만 누르세요. Google Drive 연결 창이 뜨면 허용하면 됩니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('1/4 Drive 연결 완료')


In [ ]:
!pip -q install iterative-stratification scikit-learn pandas numpy joblib
print('2/4 패키지 준비 완료')


In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/prelim_m3_exp')
if not REPO_DIR.exists():
    clone = subprocess.run([
        'git', 'clone', '--branch', 'exp/m3-char-tfidf', '--single-branch',
        'https://github.com/eigensinn-dcc2026/prelim.git', str(REPO_DIR)
    ], text=True, capture_output=True)
    if clone.returncode != 0:
        print(clone.stderr)
        raise RuntimeError('GitHub 코드 불러오기에 실패했습니다.')
print('3/4 실험 코드 준비 완료')


In [ ]:
from pathlib import Path
import subprocess, sys, json
import pandas as pd

TRAIN_LABEL_DIR = Path('/content/drive/MyDrive/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급')
if not TRAIN_LABEL_DIR.is_dir():
    raise FileNotFoundError(f'폴더를 찾지 못했습니다: {TRAIN_LABEL_DIR}')

json_files = list(TRAIN_LABEL_DIR.rglob('*.json'))
print('Training JSON 개수:', len(json_files))
if not json_files:
    raise FileNotFoundError('Training JSON이 0개입니다.')

OUTPUT_DIR = REPO_DIR / 'artifacts/m3_compare_tfidf'
result = subprocess.run([
    sys.executable,
    str(REPO_DIR / 'scripts/m3_compare_tfidf.py'),
    '--label_dir', str(TRAIN_LABEL_DIR),
    '--output_dir', str(OUTPUT_DIR),
    '--threshold', '0.5',
    '--test_size', '0.2',
    '--seed', '42'
], cwd=REPO_DIR, text=True, capture_output=True)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('실험 실행 중 오류가 발생했습니다.')

comparison = pd.read_csv(OUTPUT_DIR / 'm3_tfidf_comparison.csv')
per_label = pd.read_csv(OUTPUT_DIR / 'm3_tfidf_per_label_comparison.csv')
with (OUTPUT_DIR / 'm3_tfidf_decision.json').open('r', encoding='utf-8') as f:
    decision = json.load(f)

print('\n===== 최종 비교 =====')
display(comparison)
print('\n===== 증상별 F1 변화 =====')
display(per_label.sort_values('f1_delta', ascending=False))
print('\n===== 결론 =====')
print('Baseline Macro F1 :', round(decision['baseline_macro_f1'], 4))
print('Exp 1 Macro F1    :', round(decision['exp1_macro_f1'], 4))
print('변화량             :', round(decision['macro_f1_delta'], 4))
print('판정               :', decision['decision'])
print(decision['reason'])
print('\n4/4 완료')
